# Feature: Chatbot with RAG (Retrieval-Augmented Generation)
Goal: build a retriever (FAISS) over your MM dataset + interpretability outputs and a generation wrapper (local HF flan-t5-small by default) that returns grounded nutrition answers with citations.


In [34]:
# CELL 1: Imports & config
import os, json, math, time
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import torch
import requests
from ddgs import DDGS



print("torch version:", torch.__version__, "cuda available:", torch.cuda.is_available())

# Paths
DATA_DIR = r"C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data"
CSV_FILE = os.path.join(DATA_DIR, "MM_Food_Cleaned_Final.csv")
ART_DIR = os.path.join(DATA_DIR, "artifacts")
os.makedirs(ART_DIR, exist_ok=True)

print("CSV:", CSV_FILE)
print("ART_DIR:", ART_DIR)

# LLM server config (Qwen via llama-server)
LLAMA_SERVER_URL = "http://127.0.0.1:8080/v1/chat/completions"   # your running server
LLAMA_MODEL_NAME = "qwen-local"   # name is just a label



torch version: 2.7.0+cu118 cuda available: True
CSV: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\MM_Food_Cleaned_Final.csv
ART_DIR: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts


In [35]:
# CELL 2: Load df & build docs
df = pd.read_csv(CSV_FILE)
print("Rows:", len(df), "Cols:", df.shape[1])
display(df.head(2))

# Ensure mm_id exists
if 'mm_id' not in df.columns:
    df = df.reset_index().rename(columns={'index': 'mm_id'})

docs = []

def add_doc(doc_id, text, meta):
    if not isinstance(text, str) or text.strip() == "":
        return
    docs.append({"id": str(doc_id), "text": text.strip(), "meta": meta})

# Build docs from rows: dish_name + ingredients + per-100g nutrition
for idx, row in df.iterrows():
    mmid = int(row.get('mm_id', idx))
    dish = str(row.get('dish_name', '')).strip()
    ingredients = row.get('ingredients', '')
    per100_parts = []
    for c in ['calories_kcal_per_100g','protein_g_per_100g','fat_g_per_100g','carbohydrate_g_per_100g']:
        if c in df.columns and not pd.isna(row.get(c, np.nan)):
            per100_parts.append(f"{c.replace('_per_100g','')}: {row[c]}")
    per100_text = "; ".join(per100_parts)
    
    text = dish if dish else ""
    if isinstance(ingredients, str) and ingredients.strip():
        text += (" — Ingredients: " + ingredients) if text else ("Ingredients: " + ingredients)
    if per100_text:
        text += " — " + per100_text

    meta = {"mm_id": mmid, "dish_name": dish}
    add_doc(f"row_{mmid}", text, meta)

# Optional: interpretation doc
interp_path = os.path.join(ART_DIR, "last_meal_interpretation.json")
if os.path.exists(interp_path):
    with open(interp_path,'r',encoding='utf8') as f:
        interp = json.load(f)
    s = interp.get('summary','') + " " + " ".join([f.get('message','') for f in interp.get('flags',[])])
    add_doc("interpretation_last", s, {"source":"interpretation"})

print("Built", len(docs), "documents. Sample 3:")
docs[:3]

# Save docs
with open(os.path.join(ART_DIR, "rag_corpus.json"), "w", encoding="utf8") as f:
    json.dump(docs, f, indent=2, ensure_ascii=False)
print("Saved corpus:", os.path.join(ART_DIR, "rag_corpus.json"))


C:\Users\Aaditya\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\core\interactiveshell.py:3500: ResourceWarning: unclosed <ssl.SSLSocket fd=6724, family=2, type=1, proto=0, laddr=('192.168.0.100', 50782), raddr=('20.204.244.192', 443)>
  to_run.append((node, "exec"))


Rows: 10996 Cols: 17


,dish_name,image_filepath,submission_date,food_type,food_type.1,cooking_method,ingredients,total_grams,is_duplicate_image,calories_kcal,fat_g,protein_g,carbohydrate_g,calories_kcal_per_100g,fat_g_per_100g,protein_g_per_100g,carbohydrate_g_per_100g
0,spicy crab,preprocessed_data\images_processed\0a0c19eb67a...,2025-07-05,Restaurant food,restaurant food,stir-fried,"['crab', 'sauce', 'spices', 'vegetables']",450.0,False,500,20.0,40.0,10.0,111.111111,4.444444,8.888889,2.222222
1,steamed snails,preprocessed_data\images_processed\066b7ba6197...,2025-07-01,Homemade food,homemade food,steamed,"['dipping sauce', 'garlic', 'herbs', 'snails']",350.0,False,200,5.0,30.0,10.0,57.142857,1.428571,8.571429,2.857143


Built 10997 documents. Sample 3:
Saved corpus: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\rag_corpus.json


In [36]:
# CELL 3: Build sentence embeddings and FAISS index (better model)
EMBED_MODEL = "all-mpnet-base-v2"   # more accurate than all-MiniLM-L6-v2
print("Loading embedder:", EMBED_MODEL)
embedder = SentenceTransformer(EMBED_MODEL)

doc_texts = [d['text'] for d in docs]
print("Encoding", len(doc_texts), "documents with", EMBED_MODEL, "...")
embs = embedder.encode(doc_texts, show_progress_bar=True, convert_to_numpy=True)

# Normalize + FAISS index (inner product)
faiss.normalize_L2(embs)
dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embs)
print("Built FAISS index with", index.ntotal, "vectors. dim:", dim)

# Save artifacts
np.save(os.path.join(ART_DIR, "rag_doc_embeddings.npy"), embs)
with open(os.path.join(ART_DIR, "rag_doc_texts.json"), "w", encoding="utf8") as f:
    json.dump(doc_texts, f, ensure_ascii=False)
with open(os.path.join(ART_DIR, "rag_doc_meta.json"), "w", encoding="utf8") as f:
    json.dump([d['meta'] for d in docs], f, ensure_ascii=False)
faiss.write_index(index, os.path.join(ART_DIR, "rag_faiss.index"))
print("Saved RAG artifacts to:", ART_DIR)


Loading embedder: all-mpnet-base-v2
Encoding 10997 documents with all-mpnet-base-v2 ...


Batches:   0%|          | 0/344 [00:00<?, ?it/s]

Built FAISS index with 10997 vectors. dim: 768
Saved RAG artifacts to: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts


In [37]:
# CELL 4: Retrieval (bi-encoder) + cross-encoder reranker
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print("Loading cross-encoder reranker:", RERANKER_MODEL)
reranker = CrossEncoder(RERANKER_MODEL)

def retrieve_bi_encoder(query, top_k=100):
    """Fast initial retrieval with bi-encoder."""
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx_i in zip(D[0], I[0]):
        if idx_i < 0:
            continue
        results.append({
            "score": float(score),
            "text": doc_texts[idx_i],
            "meta": docs[idx_i]['meta'],
            "index": int(idx_i),
        })
    return results

def rerank_with_cross_encoder(query, candidates, top_k=5):
    """Rerank candidates with cross-encoder for higher accuracy."""
    if not candidates:
        return []
    texts = [c['text'] for c in candidates]
    pairs = [[query, t] for t in texts]
    scores = reranker.predict(pairs)  # higher = better
    for c, s in zip(candidates, scores):
        c['rerank_score'] = float(s)
    sorted_hits = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return sorted_hits[:top_k]

# quick sanity test
print("Retrieve test: 'Is oatmeal good for weight loss?'")
cand = retrieve_bi_encoder("Is oatmeal good for weight loss?", top_k=50)
top = rerank_with_cross_encoder("Is oatmeal good for weight loss?", cand, top_k=5)
for i, r in enumerate(top):
    print(i, r['meta'], "score_bi:", r['score'], "rerank:", r['rerank_score'])


Loading cross-encoder reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Retrieve test: 'Is oatmeal good for weight loss?'
0 {'mm_id': 8348, 'dish_name': 'oatmeal'} score_bi: 0.46669653058052063 rerank: -1.5588159561157227
1 {'mm_id': 6009, 'dish_name': 'oatmeal'} score_bi: 0.46892881393432617 rerank: -1.560733675956726
2 {'mm_id': 568, 'dish_name': 'oatmeal'} score_bi: 0.48557889461517334 rerank: -2.6417531967163086
3 {'mm_id': 10599, 'dish_name': 'oatmeal with blackcurrants'} score_bi: 0.4253900647163391 rerank: -3.3509206771850586
4 {'mm_id': 3643, 'dish_name': 'instant oatmeal'} score_bi: 0.3961673974990845 rerank: -3.4611096382141113


In [38]:
# CELL 4.5: Web Search via DDGS (English-biased, but not overly strict)

def web_search(query, n_results=5):
    """
    Perform a DDG web search and return a list of dicts with
    title, url, and snippet. Biased to English.
    """
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(
            query,
            region="us-en",          # English (US)
            safesearch="moderate",
            max_results=n_results
        ):
            title = (r.get("title") or "").strip()
            href = (r.get("href") or "").strip()
            snippet = (r.get("body") or "").strip()

            if not title and not snippet:
                continue

            results.append({
                "title": title,
                "href": href,
                "snippet": snippet
            })

            if len(results) >= n_results:
                break

    return results

# quick sanity check
test_web = web_search("Is oatmeal good for weight loss?", n_results=3)
print("Web search sample (titles):")
for w in test_web:
    print("-", w["title"])


Web search sample (titles):
- Is Oatmeal Good for Weight Loss? Nutritionists Weigh In - Prevention
- 7 benefits of oats for weight loss and how to eat it | HealthShots
- Is Your Morning Oatmeal Helping or Hurting Your Weight Loss Goals? - Health


In [39]:
# CELL 5: LLM wrapper — local Qwen via llama-server HTTP API

def llm_generate_qwen(prompt, max_tokens=200, temperature=0.2):
    """
    Call local Qwen 2.5 3B (via llama-server) using OpenAI-like /v1/chat/completions.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful nutrition assistant. "
                "Use ONLY the provided sources in the prompt. "
                "Be concise, accurate, and cautious about health advice."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    payload = {
        "model": LLAMA_MODEL_NAME,   # arbitrary label
        "messages": messages,
        "temperature": float(temperature),
        "max_tokens": int(max_tokens),
    }

    resp = requests.post(LLAMA_SERVER_URL, json=payload)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]

# unified wrapper name (for compatibility with your old code)
def llm_generate(prompt, max_tokens=200, temperature=0.2):
    return llm_generate_qwen(prompt, max_tokens=max_tokens, temperature=temperature)

# quick sanity test (short prompt)
print(llm_generate("Summarize oatmeal as a breakfast for weight loss in 1–2 sentences.", max_tokens=80))


Oatmeal can be a nutritious option for weight loss as it is rich in fiber, which can help you feel fuller for longer, potentially reducing overall calorie intake. Additionally, oats are low in fat and high in complex carbohydrates, which can provide sustained energy without the spike in blood sugar that can lead to increased hunger and snacking.


In [40]:
# ============================================================
# CELL 6: Hybrid Prompt (Dataset + Web) with clear, tagged tips
# ============================================================

from urllib.parse import urlparse  # for pretty web source names

HYBRID_RAG_PROMPT = """
You are a helpful nutrition assistant. ALWAYS answer in clear, simple ENGLISH.

You have TWO kinds of information:
1. LOCAL DATASET SOURCES (nutrition data from a structured dataset)
2. WEB SOURCES (general information from trusted websites)

Use BOTH types of sources when they are relevant.

---------------------------------------
DATASET SOURCES (D0, D1, ...):
{dataset_sources}

WEB SOURCES (W0, W1, ...):
{web_sources}
---------------------------------------

QUESTION:
{question}

INSTRUCTIONS FOR YOUR ANSWER:
- Start with 1–2 sentences that directly and clearly answer the question
  (for example: "Yes, it can be a good choice if..." or
   "It can be okay in moderation if..." etc.).
- Do NOT say a strong "No, it is not a good choice" unless the sources clearly show it is harmful.
- Use numbers or facts from the DATASET SOURCES when possible (e.g., calories, protein, fat).
- Use WEB SOURCES for general health guidance, but keep it short and evidence-based.
- After the summary, ALWAYS include a section titled:

  **Practical suggestions (with trusted sources):**

  Then give 3–5 short bullet points. Each bullet must:
  - Be a specific, actionable tip (how to prepare, portion size, toppings to choose/avoid, frequency, etc.).
  - End with a source tag in brackets, like:
    [Dataset: D0] or [Web: W1 - Prevention] or [Web: W2 - Healthline].

- If the dataset does NOT contain enough information, you may say:
  "The dataset alone does not give a full picture, so here is general advice based on web sources and common nutrition guidance:"
  and then continue with safe, general advice.
- Keep the entire answer in ENGLISH.
- Aim for 4–8 sentences in total plus the bullet list.

Answer:
"""

def build_dataset_sources_text(retrieved_docs, truncate_chars=450):
    """
    Convert dataset hits into a compact text block with labels D0, D1, ...
    """
    srcs = []
    for i, d in enumerate(retrieved_docs):
        m = d.get('meta', {})
        tag_id = f"D{i}"
        dish = m.get('dish_name') or m.get('mm_id') or f"doc_{i}"
        txt = d['text']
        if len(txt) > truncate_chars:
            txt = txt[:truncate_chars].rsplit(' ', 1)[0] + "..."
        srcs.append(f"{tag_id}: {dish} — {txt}")
    return "\n".join(srcs) if srcs else "No dataset sources found."

def build_web_sources_text(web_docs, truncate_chars=300):
    """
    Convert web results into a compact text block with labels W0, W1, ...
    Include domain so LLM can refer to 'trusted sources' by name.
    """
    srcs = []
    for i, w in enumerate(web_docs):
        title = (w.get("title") or "").strip() or "No title"
        snip = (w.get("snippet") or "").strip()
        if len(snip) > truncate_chars:
            snip = snip[:truncate_chars].rsplit(' ', 1)[0] + "..."
        href = (w.get("href") or "").strip()
        domain = urlparse(href).netloc or "unknown site"
        label = f"W{i}"
        # Example: W0: Is Oatmeal Good for Weight Loss? (site: www.health.com) — snippet...
        srcs.append(f"{label}: {title} (site: {domain}) — {snip}")
    return "\n".join(srcs) if srcs else "No web sources found."

def build_hybrid_prompt(question, ds_docs, web_docs,
                        ds_truncate=450, web_truncate=300):
    ds_text = build_dataset_sources_text(ds_docs, truncate_chars=ds_truncate)
    web_text = build_web_sources_text(web_docs, truncate_chars=web_truncate)
    return HYBRID_RAG_PROMPT.format(
        dataset_sources=ds_text,
        web_sources=web_text,
        question=question
    )

def rag_answer(question,
               top_k=5,      # final dataset docs after rerank
               bi_k=100,     # initial dataset candidates
               web_k=5,      # number of web results (0 = disable web)
               gen_max_tokens=320):
    # 1) Dataset retrieval
    ds_candidates = retrieve_bi_encoder(question, top_k=bi_k)
    ds_hits = rerank_with_cross_encoder(question, ds_candidates, top_k=top_k)

    # 2) Web search
    web_hits = web_search(question, n_results=web_k) if web_k > 0 else []

    # 3) Build combined prompt
    prompt = build_hybrid_prompt(question, ds_hits, web_hits)

    # 4) Generate answer with Qwen
    answer = llm_generate(prompt, max_tokens=gen_max_tokens, temperature=0.2)

    # 5) Prepare compact dataset summary (for debugging / UI)
    used_dataset = [
        {
            "rank": i,
            "score": r.get('rerank_score', None),
            "dish_name": r['meta'].get('dish_name'),
            "mm_id": r['meta'].get('mm_id')
        }
        for i, r in enumerate(ds_hits)
    ]

    return {
        "question": question,
        "answer": answer.strip(),
        "dataset_used": ds_hits,
        "web_used": web_hits,
        "dataset_used_summary": used_dataset,
    }

# quick demo
sample_q = "Is oatmeal a good choice for breakfast if I want to lose weight?"
print("Question:", sample_q)
res = rag_answer(sample_q, top_k=4, bi_k=100, web_k=5, gen_max_tokens=320)


print("\nAnswer:\n", res['answer'])

print("\nDataset sources used (summary):")
for u in res['dataset_used_summary']:
    print("-", u)

print("\nWeb sources used (titles):")
for w in res['web_used']:
    print("-", w["title"])


Question: Is oatmeal a good choice for breakfast if I want to lose weight?


C:\Users\Aaditya\AppData\Local\Programs\Python\Python312\Lib\collections\__init__.py:449: ResourceWarning: unclosed <ssl.SSLSocket fd=6240, family=2, type=1, proto=0, laddr=('192.168.0.100', 61667), raddr=('20.204.244.192', 443)>
  result = tuple_new(cls, iterable)



Answer:
 Yes, it can be a good choice if prepared and consumed in moderation. Oatmeal is a nutritious breakfast option that can provide sustained energy and fiber, which are beneficial for weight loss efforts. It contains carbohydrates, protein, and fiber, which can help keep you full and stabilize blood sugar levels.

**Practical suggestions (with trusted sources):**

- **Choose unsweetened oatmeal:** [Web: W1 - Beauty and Care News] suggests avoiding added sugars to keep the calorie count lower.
- **Use a portion of milk or water:** [Dataset: D0] indicates that using 1/2 cup of water can reduce the calorie content of oatmeal.
- **Add fruits and nuts:** [Web: W2 - Weight Loss Tips!] recommends including fruits and nuts for added nutrients and fiber.
- **Monitor portion sizes:** [Dataset: D0] shows that 1/2 cup of oatmeal contains about 150 calories, which is a reasonable portion for breakfast.
- **Limit toppings:** [Web: W1 - Beauty and Care News] advises limiting toppings to avoid u

In [41]:
# CELL 7: Save RAG config for Streamlit / reuse (updated for web)
rag_cfg = {
    "embed_model": EMBED_MODEL,
    "faiss_index": os.path.join(ART_DIR, "rag_faiss.index"),
    "doc_texts": os.path.join(ART_DIR, "rag_doc_texts.json"),
    "doc_meta": os.path.join(ART_DIR, "rag_doc_meta.json"),
    "llm_backend": "llama_server",
    "llama_server_url": LLAMA_SERVER_URL,
    "llama_model_name": LLAMA_MODEL_NAME,
    "web_backend": "duckduckgo",          # NEW
    "use_web": True                       # NEW
}
cfg_path = os.path.join(ART_DIR, "rag_config.json")
with open(cfg_path, "w", encoding="utf8") as f:
    json.dump(rag_cfg, f, indent=2)
print("Saved RAG config to:", cfg_path)


Saved RAG config to: C:\Users\Aaditya\Desktop\NutriSage_Local\preprocessed_data\artifacts\rag_config.json
